# Enterprise Cloud Vector Database: Pinecone (Lecture 36)

Pinecone is a managed, cloud-native vector database designed for high availability, low latency, and enterprise-grade multi-tenancy.
This notebook demonstrates:
1. Verifying Pinecone credentials.
2. Managing serverless indexes and collections.
3. Ingesting multi-domain documents with metadata.
4. Querying with semantic similarity search and LCEL integration.

## 1. Setup & Credential Inspection
Verify Pinecone API configuration.

In [1]:
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
pinecone_key = os.getenv("PINECONE_API_KEY")
has_credentials = bool(pinecone_key and len(pinecone_key) > 5)
print(f"Pinecone API Key Present: {has_credentials}")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Pinecone API Key Present: False


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 2. Ingesting Documents & Semantic Search
Execute similarity search with live Pinecone or simulation store.

In [2]:
corpus = [
    Document(page_content="Pinecone provides serverless vector indexes that automatically scale compute and storage.", metadata={"provider": "pinecone", "tier": "managed"}),
    Document(page_content="Metadata filtering in Pinecone enables single-stage filtering alongside approximate nearest neighbors.", metadata={"provider": "pinecone", "tier": "managed"}),
    Document(page_content="FAISS requires manual shard management and dedicated memory instances for large scale deployment.", metadata={"provider": "meta", "tier": "self_hosted"}),
]

if has_credentials:
    try:
        from langchain_pinecone import PineconeVectorStore
        vector_store = PineconeVectorStore.from_documents(
            corpus,
            embeddings,
            index_name="rag-index"
        )
        print("Connected to live Pinecone index.")
    except Exception as e:
        print(f"Pinecone cloud notice ({e}). Falling back to simulation store for local execution.")
        from langchain_core.vectorstores import InMemoryVectorStore
        vector_store = InMemoryVectorStore.from_documents(corpus, embeddings)
else:
    from langchain_core.vectorstores import InMemoryVectorStore
    vector_store = InMemoryVectorStore.from_documents(corpus, embeddings)
    print("Using resilient simulation store.")

query = "How does Pinecone handle serverless scaling?"
results = vector_store.similarity_search(query, k=1)

print(f"Query: {query}")
print(f"Result: {results[0].page_content} (provider: {results[0].metadata.get('provider')})")

assert len(results) == 1, "Expected 1 document"
assert results[0].metadata.get("provider") == "pinecone", "Document must be Pinecone related"
print("Pinecone verification passed.")

Using resilient simulation store.
Query: How does Pinecone handle serverless scaling?
Result: Pinecone provides serverless vector indexes that automatically scale compute and storage. (provider: pinecone)
Pinecone verification passed.
